# 01 建库 + 知识库入库 + 检索验证

本 notebook 完成：数据库建表、RAG 入库流水线、GraphRAG 检索验证。

| Cell | 内容 |
|---|---|
| 1 | 连接 PostgreSQL，create_all() 建表 + HNSW 索引 |
| 2 | 基本 CRUD 验证 |
| 3 | md 切分测试 |
| 4 | 向量化测试 |
| 5 | 入库示例知识库 |
| 6 | 查看知识图谱 |
| 7 | 向量检索测试 |
| 8 | GraphRAG 检索测试 |
| 9 | 边类型优先级测试 |
| 10 | 空检索兜底测试 |

In [ ]:
# Cell 1: 连接数据库 + 建表 + HNSW 索引
from teaching_agent.db import init_db, async_engine
from sqlalchemy import text

await init_db()
print("✓ 表创建成功")

# 创建 HNSW 向量索引（cosine 距离）
async with async_engine.begin() as conn:
    await conn.execute(text(
        "CREATE INDEX IF NOT EXISTS idx_chunk_embedding_hnsw "
        "ON chunk USING hnsw (embedding vector_cosine_ops)"
    ))
print("✓ HNSW 索引创建成功")

In [ ]:
# Cell 2: 基本 CRUD 验证
from teaching_agent.models import Session, StudentProfile
from teaching_agent.db import async_session_factory
from datetime import datetime, timezone

async with async_session_factory() as session:
    # 清理可能残留的测试数据
    existing = await session.get(Session, "test-session-1")
    if existing:
        await session.delete(existing)
    existing_p = await session.get(StudentProfile, "profile-1")
    if existing_p:
        await session.delete(existing_p)
    await session.commit()

    # 创建 demo session
    s = Session(
        id="test-session-1", user_id="demo-student", type="free",
        status="active", started_at=datetime.now(timezone.utc),
    )
    session.add(s)

    # 创建 demo 画像
    p = StudentProfile(
        id="profile-1", user_id="demo-student",
        summary="新学生", weak_points=[], learning_style="未知",
    )
    session.add(p)
    await session.commit()
    print("✓ CRUD 验证成功")

    # 查询验证
    result = await session.get(Session, "test-session-1")
    print(f"✓ 查询: type={result.type}, status={result.status}")

    # 清理测试数据
    await session.delete(result)
    await session.delete(await session.get(StudentProfile, "profile-1"))
    await session.commit()
    print("✓ 清理完成")

In [ ]:
# Cell 3: md 切分测试
from teaching_agent.rag.chunker import chunk_markdown
from teaching_agent.knowledge.sample_data import COURSE_DOCUMENTS

doc = COURSE_DOCUMENTS[0]
chunks = chunk_markdown(doc["content"])
print(f"文档: {doc['filename']}")
print(f"切分出 {len(chunks)} 个 chunk:")
for c in chunks[:10]:
    print(f"  [{c.chunk_index}] {c.heading_path} ({len(c.content)} 字符)")
    print(f"      {c.content[:80]}...")
if len(chunks) > 10:
    print(f"  ... 还有 {len(chunks) - 10} 个 chunk")

In [ ]:
# Cell 4: 向量化测试
from teaching_agent.rag.embedder import embed_texts

texts = [c.content for c in chunks[:3]]
vectors = await embed_texts(texts)
print(f"向量化 {len(vectors)} 个 chunk")
print(f"维度: {len(vectors[0])}")
print(f"前5维: {vectors[0][:5]}")

In [ ]:
# Cell 5: 入库示例知识库（切分 -> 向量化 -> 写入 pgvector -> LLM 知识图谱分析）
# 注意：此 cell 会对每个 chunk 调用 LLM 进行知识图谱分析，4 篇文档约 200+ chunks，耗时较长
from teaching_agent.rag.pipeline import ingest_document
from teaching_agent.knowledge.sample_data import COURSE_DOCUMENTS
from teaching_agent.db import async_engine
from sqlalchemy import text

# 清空旧数据（重新入库时用）
async with async_engine.begin() as conn:
    await conn.execute(text("DELETE FROM chunkknowledgepoint"))
    await conn.execute(text("DELETE FROM knowledgeedge"))
    await conn.execute(text("DELETE FROM knowledgepoint"))
    await conn.execute(text("DELETE FROM chunk"))
    await conn.execute(text("DELETE FROM kbdocument"))

# 入库 4 篇课程文档
for doc in COURSE_DOCUMENTS:
    await ingest_document(
        filename=doc["filename"],
        content=doc["content"],
        scope=doc["scope"],
    )

In [ ]:
# Cell 6: 查看知识图谱全貌（节点 + 边）
from teaching_agent.db import async_session_factory
from teaching_agent.rag.graph import get_graph_view

async with async_session_factory() as session:
    graph = await get_graph_view(session)

print(f"知识点（{len(graph['nodes'])} 个）:")
for node in graph["nodes"]:
    print(f"  - {node['name']}: {node['description']}")

print(f"\n边（{len(graph['edges'])} 条）:")
for edge in graph["edges"]:
    print(f"  {edge['source']} --{edge['relation_type']}--> {edge['target']}")

In [ ]:
# Cell 7: 向量检索测试 -- 不同 query 的 top-5
from teaching_agent.rag.retriever import graphrag_search

async with async_session_factory() as session:
    for query in ["TCP三次握手过程", "UDP和TCP的区别", "拥塞控制算法", "物理层的作用"]:
        chunks = await graphrag_search(session, query, "free")
        print(f"\n{'='*50}")
        print(f"Query: {query}")
        print(f"命中 {len(chunks)} 个 chunk:")
        for c in chunks:
            print(f"  [{c['source']}] dist={c['dist']:.4f} | {c['heading_path']}")

In [ ]:
# Cell 8: GraphRAG 检索测试 -- 验证图遍历扩展效果
async with async_session_factory() as session:
    chunks = await graphrag_search(session, "数据链路层封装成帧", "free")
    print("GraphRAG 检索结果:")
    for c in chunks:
        print(f"  [{c['source']}] dist={c['dist']:.4f} | {c['heading_path']}")
        print(f"    {c['content'][:60]}...")

In [ ]:
# Cell 9: 边类型优先级测试 -- 查看图遍历是否正确扩展了 prerequisite/contrast/related 的 chunk
async with async_session_factory() as session:
    chunks = await graphrag_search(session, "CSMA/CD协议", "free")
    graph_chunks = [c for c in chunks if c["source"].startswith("graph")]
    print(f"图扩展结果 ({len(graph_chunks)} 个):")
    for c in graph_chunks:
        print(f"  {c['source']} | {c['heading_path']}")
    if not graph_chunks:
        print("  （本次检索无图扩展结果，可能知识点间无边或已被向量结果覆盖）")

In [ ]:
# Cell 10: 空检索兜底测试
from teaching_agent.rag.retriever import format_chunks

async with async_session_factory() as session:
    chunks = await graphrag_search(session, "量子力学原理", "free")
    result = format_chunks(chunks)
    print(result)